# WoWAH Staging Clean Export

Bu notebook, ham WoWAH TXT dosyalarini parse eder, anomali satirlarini ayirir ve full clean staging Parquet part ciktilarini uretir.

In [1]:
import csv
import math
import re
from pathlib import Path

import duckdb
import pandas as pd

print('DuckDB ve pandas hazir.')


DuckDB ve pandas hazir.


In [2]:
def first_existing_path(candidates, *, is_dir=False):
    for path in candidates:
        if not path.exists():
            continue
        if is_dir and not path.is_dir():
            continue
        return path
    return None


raw_root_candidates = [
    Path('/Users/osmanorka/Downloads/wowah/WoWAH'),
    Path.cwd() / 'wowah/WoWAH',
    Path.cwd().parent / 'wowah/WoWAH',
    Path.cwd() / 'data/raw/wowah/WoWAH',
    Path.cwd().parent / 'data/raw/wowah/WoWAH',
]

raw_root = first_existing_path(raw_root_candidates, is_dir=True)
if raw_root is None:
    raise FileNotFoundError('WoWAH klasoru bulunamadi.')

processed_dir = Path.cwd() / 'data/processed'
preview_clean_dir = processed_dir / 'preview_stg_wowah_events_clean_parts'
preview_anomaly_dir = processed_dir / 'preview_stg_wowah_event_anomalies_parts'
clean_parts_dir = processed_dir / 'stg_wowah_events_clean_parts'
anomaly_parts_dir = processed_dir / 'stg_wowah_event_anomalies_parts'

for directory in [preview_clean_dir, preview_anomaly_dir, clean_parts_dir, anomaly_parts_dir]:
    directory.mkdir(parents=True, exist_ok=True)

print('Raw root:', raw_root)
print('Preview clean dir:', preview_clean_dir)
print('Preview anomaly dir:', preview_anomaly_dir)
print('Clean parts dir:', clean_parts_dir)
print('Anomaly parts dir:', anomaly_parts_dir)


Raw root: /Users/osmanorka/Downloads/wowah/WoWAH
Preview clean dir: /Users/osmanorka/PlayerPulse-LiveOps/notebooks/data/processed/preview_stg_wowah_events_clean_parts
Preview anomaly dir: /Users/osmanorka/PlayerPulse-LiveOps/notebooks/data/processed/preview_stg_wowah_event_anomalies_parts
Clean parts dir: /Users/osmanorka/PlayerPulse-LiveOps/notebooks/data/processed/stg_wowah_events_clean_parts
Anomaly parts dir: /Users/osmanorka/PlayerPulse-LiveOps/notebooks/data/processed/stg_wowah_event_anomalies_parts


In [3]:
record_line_pattern = re.compile(r'"([^"]*)"')


def extract_record_strings(storage_block):
    records = []
    for line in storage_block.splitlines():
        match = record_line_pattern.search(line)
        if match:
            records.append(match.group(1))
    return records


def parse_wowah_file(file_path):
    file_path = Path(file_path)
    text = file_path.read_text(encoding='utf-8', errors='ignore')

    storage_match = re.search(
        r'Persistent_Storage\s*=\s*\{(.*?)^\s*\}',
        text,
        flags=re.DOTALL | re.MULTILINE,
    )
    if storage_match is None:
        raise ValueError(f'Persistent_Storage blogu bulunamadi: {file_path}')

    records = extract_record_strings(storage_match.group(1))
    rows = []

    for raw_record in records:
        parts = next(csv.reader([raw_record], skipinitialspace=True))
        if len(parts) != 11:
            continue
        rows.append([part.strip() for part in parts])

    columns = [
        'realm_or_source_id',
        'observed_at',
        'player_or_group_id',
        'avatar_id',
        'guild_id',
        'level',
        'race',
        'character_class',
        'zone',
        'guild_member_flag',
        'unknown_flag',
    ]

    df = pd.DataFrame(rows, columns=columns)
    if df.empty:
        return df

    df = df.replace({'': pd.NA})
    df['guild_member_flag_raw'] = df['guild_member_flag'].astype('string').str.strip().replace({'': pd.NA})

    numeric_columns = [
        'realm_or_source_id',
        'player_or_group_id',
        'avatar_id',
        'guild_id',
        'level',
        'unknown_flag',
    ]
    for column in numeric_columns:
        df[column] = pd.to_numeric(df[column], errors='coerce').astype('Int64')

    df['observed_at'] = pd.to_datetime(
        df['observed_at'],
        format='%m/%d/%y %H:%M:%S',
        errors='coerce',
    )

    flag_key = df['guild_member_flag_raw'].str.lower()
    df['guild_member_flag'] = pd.Series(pd.NA, index=df.index, dtype='boolean')
    df.loc[flag_key == 'yes', 'guild_member_flag'] = True
    df.loc[flag_key == 'no', 'guild_member_flag'] = False

    return df



In [4]:
VALID_RACES = {'Orc', 'Tauren', 'Troll', 'Undead', 'Blood Elf'}
VALID_CLASSES = {
    'Warrior',
    'Hunter',
    'Shaman',
    'Rogue',
    'Mage',
    'Priest',
    'Warlock',
    'Druid',
    'Paladin',
    'Death Knight',
}


def attach_reason(series, mask, reason):
    current = series.loc[mask].fillna('')
    series.loc[mask] = current.where(current == '', current + '|') + reason


def split_clean_and_anomalies(df):
    staged = df.copy()
    staged['race_raw'] = staged['race'].astype('string')
    staged['character_class_raw'] = staged['character_class'].astype('string')
    staged['race'] = staged['race_raw'].str.strip()
    staged['character_class'] = staged['character_class_raw'].str.strip()
    staged['zone'] = staged['zone'].astype('string').str.strip()
    staged['anomaly_reason'] = pd.Series(pd.NA, index=staged.index, dtype='string')

    rules = [
        ('missing_observed_at', staged['observed_at'].isna()),
        ('missing_avatar_id', staged['avatar_id'].isna()),
        ('invalid_level', staged['level'].isna() | ~staged['level'].between(1, 80)),
        ('invalid_race', ~staged['race'].isin(VALID_RACES)),
        ('invalid_character_class', ~staged['character_class'].isin(VALID_CLASSES)),
        ('missing_zone', staged['zone'].isna() | (staged['zone'] == '')),
        (
            'duplicate_observed_at_avatar_id_source_file',
            staged.duplicated(subset=['observed_at', 'avatar_id', 'source_file'], keep='first'),
        ),
    ]

    for reason, mask in rules:
        attach_reason(staged['anomaly_reason'], mask, reason)

    anomaly_mask = staged['anomaly_reason'].notna()

    clean_df = staged.loc[~anomaly_mask, [
        'realm_or_source_id',
        'observed_at',
        'player_or_group_id',
        'avatar_id',
        'guild_id',
        'level',
        'race',
        'character_class',
        'zone',
        'guild_member_flag',
        'unknown_flag',
        'source_file',
    ]].reset_index(drop=True)

    anomaly_df = staged.loc[anomaly_mask, [
        'realm_or_source_id',
        'observed_at',
        'player_or_group_id',
        'avatar_id',
        'guild_id',
        'level',
        'race_raw',
        'character_class_raw',
        'zone',
        'guild_member_flag_raw',
        'guild_member_flag',
        'unknown_flag',
        'source_file',
        'anomaly_reason',
    ]].reset_index(drop=True)

    return clean_df, anomaly_df



In [5]:
txt_files = sorted(raw_root.rglob('*.txt'))
print('Toplam txt dosyasi:', len(txt_files))
for file in txt_files[:10]:
    print(file)


Toplam txt dosyasi: 138084
/Users/osmanorka/Downloads/wowah/WoWAH/2006_01_03/2006-01-01/00-03-56.txt
/Users/osmanorka/Downloads/wowah/WoWAH/2006_01_03/2006-01-01/00-13-43.txt
/Users/osmanorka/Downloads/wowah/WoWAH/2006_01_03/2006-01-01/00-23-48.txt
/Users/osmanorka/Downloads/wowah/WoWAH/2006_01_03/2006-01-01/00-33-48.txt
/Users/osmanorka/Downloads/wowah/WoWAH/2006_01_03/2006-01-01/00-43-42.txt
/Users/osmanorka/Downloads/wowah/WoWAH/2006_01_03/2006-01-01/00-53-45.txt
/Users/osmanorka/Downloads/wowah/WoWAH/2006_01_03/2006-01-01/01-03-43.txt
/Users/osmanorka/Downloads/wowah/WoWAH/2006_01_03/2006-01-01/01-13-47.txt
/Users/osmanorka/Downloads/wowah/WoWAH/2006_01_03/2006-01-01/01-23-45.txt
/Users/osmanorka/Downloads/wowah/WoWAH/2006_01_03/2006-01-01/01-33-41.txt


In [6]:
sample_files = txt_files[:10]
sample_dfs = []

for file in sample_files:
    df = parse_wowah_file(file)
    if df.empty:
        continue
    df['source_file'] = str(file.relative_to(raw_root))
    sample_dfs.append(df)

df_sample_raw = pd.concat(sample_dfs, ignore_index=True)
df_sample_clean, df_sample_anomalies = split_clean_and_anomalies(df_sample_raw)

print('Ham satir:', len(df_sample_raw))
print('Temiz satir:', len(df_sample_clean))
print('Anomali satir:', len(df_sample_anomalies))
print('Duplicate observed_at + avatar_id:', df_sample_clean.duplicated(subset=['observed_at', 'avatar_id']).sum())
print('Unique avatar_id:', df_sample_clean['avatar_id'].nunique())


Ham satir: 4057
Temiz satir: 4057
Anomali satir: 0
Duplicate observed_at + avatar_id: 0
Unique avatar_id: 597


In [7]:
print(df_sample_anomalies['anomaly_reason'].value_counts())
df_sample_anomalies[
    ['observed_at', 'player_or_group_id', 'avatar_id', 'race_raw', 'character_class_raw', 'zone', 'source_file', 'anomaly_reason']
].head(20)


Series([], Name: count, dtype: Int64)


,observed_at,player_or_group_id,avatar_id,race_raw,character_class_raw,zone,source_file,anomaly_reason


In [8]:
def write_parquet_with_duckdb(df, output_path, view_name):
    output_path = Path(output_path)
    temp_output_path = output_path.with_suffix(output_path.suffix + '.tmp')

    con = duckdb.connect()
    try:
        con.register(view_name, df)
        con.execute(f"COPY {view_name} TO '{temp_output_path}' (FORMAT PARQUET)")
    finally:
        con.close()

    temp_output_path.replace(output_path)


def clear_parquet_dir(directory):
    directory = Path(directory)
    directory.mkdir(parents=True, exist_ok=True)
    for pattern in ('*.parquet', '*.parquet.tmp'):
        for file in directory.glob(pattern):
            file.unlink()


def export_clean_batches(files, clean_output_dir, anomaly_output_dir, batch_size=500, start_batch=0, max_batches=None):
    clean_output_dir = Path(clean_output_dir)
    anomaly_output_dir = Path(anomaly_output_dir)
    clean_output_dir.mkdir(parents=True, exist_ok=True)
    anomaly_output_dir.mkdir(parents=True, exist_ok=True)

    total_files = len(files)
    num_batches = math.ceil(total_files / batch_size)
    stop_batch = num_batches if max_batches is None else min(num_batches, start_batch + max_batches)

    summaries = []
    failures = []

    for batch_idx in range(start_batch, stop_batch):
        start = batch_idx * batch_size
        end = min(start + batch_size, total_files)
        batch_files = files[start:end]
        raw_batch = []

        print(f'Batch {batch_idx + 1}/{num_batches} isleniyor. Dosya {start} - {end - 1}')

        for file in batch_files:
            try:
                df = parse_wowah_file(file)
                if df.empty:
                    continue
                df['source_file'] = str(file.relative_to(raw_root))
                raw_batch.append(df)
            except Exception as exc:
                failures.append({'file': str(file), 'error': str(exc)})

        if not raw_batch:
            continue

        batch_raw_df = pd.concat(raw_batch, ignore_index=True)
        batch_clean_df, batch_anomaly_df = split_clean_and_anomalies(batch_raw_df)

        clean_output_path = clean_output_dir / f'stg_wowah_events_clean_part_{batch_idx:04d}.parquet'
        anomaly_output_path = anomaly_output_dir / f'stg_wowah_event_anomalies_part_{batch_idx:04d}.parquet'

        if not batch_clean_df.empty:
            write_parquet_with_duckdb(batch_clean_df, clean_output_path, 'batch_clean_df_view')
        if not batch_anomaly_df.empty:
            write_parquet_with_duckdb(batch_anomaly_df, anomaly_output_path, 'batch_anomaly_df_view')

        summaries.append({
            'batch_idx': batch_idx,
            'file_count': len(batch_files),
            'raw_rows': len(batch_raw_df),
            'clean_rows': len(batch_clean_df),
            'anomaly_rows': len(batch_anomaly_df),
            'clean_output_path': str(clean_output_path) if not batch_clean_df.empty else pd.NA,
            'anomaly_output_path': str(anomaly_output_path) if not batch_anomaly_df.empty else pd.NA,
        })

    return pd.DataFrame(summaries), pd.DataFrame(failures)



In [9]:
clear_parquet_dir(preview_clean_dir)
clear_parquet_dir(preview_anomaly_dir)

preview_summary, preview_failures = export_clean_batches(
    files=txt_files[:10],
    clean_output_dir=preview_clean_dir,
    anomaly_output_dir=preview_anomaly_dir,
    batch_size=10,
    max_batches=1,
)

preview_summary


Batch 1/1 isleniyor. Dosya 0 - 9


,batch_idx,file_count,raw_rows,clean_rows,anomaly_rows,clean_output_path,anomaly_output_path
0,0,10,4057,4057,0,/Users/osmanorka/PlayerPulse-LiveOps/notebooks...,<NA>


In [10]:
clear_parquet_dir(clean_parts_dir)
clear_parquet_dir(anomaly_parts_dir)

export_summary, export_failures = export_clean_batches(
    files=txt_files,
    clean_output_dir=clean_parts_dir,
    anomaly_output_dir=anomaly_parts_dir,
    batch_size=500,
)

print('Export tamamlandi.')
print('Temiz batch sayisi:', len(export_summary))
print('Hata sayisi:', len(export_failures))
export_summary.head()


Batch 1/277 isleniyor. Dosya 0 - 499
Batch 2/277 isleniyor. Dosya 500 - 999
Batch 3/277 isleniyor. Dosya 1000 - 1499
Batch 4/277 isleniyor. Dosya 1500 - 1999
Batch 5/277 isleniyor. Dosya 2000 - 2499
Batch 6/277 isleniyor. Dosya 2500 - 2999
Batch 7/277 isleniyor. Dosya 3000 - 3499
Batch 8/277 isleniyor. Dosya 3500 - 3999
Batch 9/277 isleniyor. Dosya 4000 - 4499
Batch 10/277 isleniyor. Dosya 4500 - 4999
Batch 11/277 isleniyor. Dosya 5000 - 5499
Batch 12/277 isleniyor. Dosya 5500 - 5999
Batch 13/277 isleniyor. Dosya 6000 - 6499
Batch 14/277 isleniyor. Dosya 6500 - 6999
Batch 15/277 isleniyor. Dosya 7000 - 7499
Batch 16/277 isleniyor. Dosya 7500 - 7999
Batch 17/277 isleniyor. Dosya 8000 - 8499
Batch 18/277 isleniyor. Dosya 8500 - 8999
Batch 19/277 isleniyor. Dosya 9000 - 9499
Batch 20/277 isleniyor. Dosya 9500 - 9999
Batch 21/277 isleniyor. Dosya 10000 - 10499
Batch 22/277 isleniyor. Dosya 10500 - 10999
Batch 23/277 isleniyor. Dosya 11000 - 11499
Batch 24/277 isleniyor. Dosya 11500 - 11999

,batch_idx,file_count,raw_rows,clean_rows,anomaly_rows,clean_output_path,anomaly_output_path
0,0,500,136020,136020,0,/Users/osmanorka/PlayerPulse-LiveOps/notebooks...,<NA>
1,1,500,164343,164343,0,/Users/osmanorka/PlayerPulse-LiveOps/notebooks...,<NA>
2,2,500,131249,131249,0,/Users/osmanorka/PlayerPulse-LiveOps/notebooks...,<NA>
3,3,500,146928,146928,0,/Users/osmanorka/PlayerPulse-LiveOps/notebooks...,<NA>
4,4,500,145669,145669,0,/Users/osmanorka/PlayerPulse-LiveOps/notebooks...,<NA>


In [11]:
full_clean_source_glob = str(clean_parts_dir / '*.parquet')
con = duckdb.connect()
con.execute(
    f"CREATE OR REPLACE VIEW stg_wowah_events_full_clean AS SELECT * FROM read_parquet('{full_clean_source_glob}')"
)

full_validation_overview = con.execute(
    '''
    WITH base AS (
        SELECT *
        FROM stg_wowah_events_full_clean
    )
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT avatar_id) AS unique_avatar_id,
        MIN(observed_at) AS min_observed_at,
        MAX(observed_at) AS max_observed_at,
        COUNT(DISTINCT CAST(observed_at AS DATE)) AS distinct_activity_dates,
        COUNT(*) - COUNT(DISTINCT (CAST(observed_at AS VARCHAR) || '|' || CAST(avatar_id AS VARCHAR))) AS duplicate_observed_at_avatar_id_count,
        AVG(CASE WHEN observed_at IS NULL THEN 1.0 ELSE 0.0 END) AS observed_at_null_rate,
        AVG(CASE WHEN avatar_id IS NULL THEN 1.0 ELSE 0.0 END) AS avatar_id_null_rate,
        AVG(CASE WHEN level IS NULL THEN 1.0 ELSE 0.0 END) AS level_null_rate,
        AVG(CASE WHEN race IS NULL THEN 1.0 ELSE 0.0 END) AS race_null_rate,
        AVG(CASE WHEN character_class IS NULL THEN 1.0 ELSE 0.0 END) AS character_class_null_rate,
        AVG(CASE WHEN zone IS NULL THEN 1.0 ELSE 0.0 END) AS zone_null_rate,
        MIN(level) AS level_min,
        MAX(level) AS level_max
    FROM base
    '''
).df()

top_race_values = con.execute(
    '''
    SELECT race, COUNT(*) AS row_count
    FROM stg_wowah_events_full_clean
    GROUP BY 1
    ORDER BY 2 DESC, 1 ASC
    LIMIT 10
    '''
).df()

top_character_class_values = con.execute(
    '''
    SELECT character_class, COUNT(*) AS row_count
    FROM stg_wowah_events_full_clean
    GROUP BY 1
    ORDER BY 2 DESC, 1 ASC
    LIMIT 10
    '''
).df()

top_zone_values = con.execute(
    '''
    SELECT zone, COUNT(*) AS row_count
    FROM stg_wowah_events_full_clean
    GROUP BY 1
    ORDER BY 2 DESC, 1 ASC
    LIMIT 10
    '''
).df()

full_validation_overview


,total_rows,unique_avatar_id,min_observed_at,max_observed_at,distinct_activity_dates,duplicate_observed_at_avatar_id_count,observed_at_null_rate,avatar_id_null_rate,level_null_rate,race_null_rate,character_class_null_rate,zone_null_rate,level_min,level_max
0,36513647,91064,2005-12-31 23:59:46,2009-01-10 05:08:59,1083,9459,0.0,0.0,0.0,0.0,0.0,0.0,1,80


In [12]:
print('Top 10 race values')
print(top_race_values)
print('\nTop 10 character_class values')
print(top_character_class_values)
print('\nTop 10 zone values')
print(top_zone_values)


Top 10 race values
        race  row_count
0     Undead   11365578
1     Tauren    7985224
2  Blood Elf    7756519
3      Troll    5464194
4        Orc    3901512
5       547人      29255
6       373族      11353
7       3033          6
8      27410          3
9     74622妖          3

Top 10 character_class values
  character_class  row_count
0          Hunter    5753267
1            Mage    5370220
2         Warrior    5262739
3         Warlock    4215018
4          Priest    4093239
5           Rogue    3917603
6          Shaman    2678181
7           Druid    2578147
8         Paladin    2369781
9    Death Knight     275250

Top 10 zone values
                 zone  row_count
0           Orgrimmar    3091805
1      Shattrath City    1892294
2         The Barrens    1264010
3  Stranglethorn Vale    1063264
4     Terokkar Forest     983116
5      Alterac Valley     979136
6        Arathi Basin     965258
7  Hellfire Peninsula     952413
8             Nagrand     931646
9            Kara

In [13]:
distinct_activity_dates = int(full_validation_overview.loc[0, 'distinct_activity_dates'])
if distinct_activity_dates <= 2:
    raise ValueError('Full clean staging hala preview-like gorunuyor: distinct activity dates <= 2.')

print('Sanity check gecti. Distinct activity dates:', distinct_activity_dates)


Sanity check gecti. Distinct activity dates: 1083
